# 실험 02: 기억력을 가진 Agent (Memory)

## 1. 실험 제목과 목표
- **제목**: 어제 한 일도 기억하는 비서 만들기
- **목표**: 교안 두 번째 슬라이드의 Agent 구성 요소 중 **Memory(이전 대화와 작업 상태 저장)** 기능을 추가합니다. Agent가 단순한 단발성 질의응답을 넘어 과거의 대화와 자신이 썼던 Tool의 결과를 기억하며 문맥을 유지하도록 만듭니다.

## 2. 실험 개요
1. **실험 1 (건망증 Agent)**: 일반 Agent에게 이름을 알려주고 툴을 쓰게 한 뒤, 다시 이름을 물어보면 까먹는 현상 확인.
2. **실험 2 (기억 장치 부착)**: LangChain의 `RunnableWithMessageHistory`를 사용하여, 세션 ID(대화방 번호)별로 대화 기록을 관리하고 이를 Agent에게 주입하는 방법 실습.

## 3. 라이브러리 import

In [1]:
import os
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_classic.agents import create_tool_calling_agent, AgentExecutor
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

## 4. 환경 변수 로드 및 기본 Agent 세팅

In [2]:
load_dotenv()
llm = ChatOpenAI(model="gpt-4o-mini")

@tool
def get_word_length(word: str) -> int:
    """단어의 글자 수를 반환합니다."""
    return len(word)

tools = [get_word_length]

# 매우 중요: 프롬프트에 'chat_history'라는 공간을 뚫어두어야 과거 기억이 들어갈 수 있습니다.
prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 똑똑한 비서입니다."),
    MessagesPlaceholder(variable_name="chat_history"), # 이전 대화가 들어갈 자리
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

agent = create_tool_calling_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools)

## 5. 실험 1: 건망증 증세 확인
일반 `AgentExecutor`는 호출될 때마다 백지 상태에서 시작합니다.

In [3]:
print("=== [1. 일반 Agent 테스트] ===")

# 첫 번째 대화
res1 = agent_executor.invoke({
    "input": "내 이름은 '슈퍼맨'이야. 내 이름의 글자 수가 몇 개인지 툴을 써서 계산해줘.",
    "chat_history": [] # 비어있는 상태로 전달
})
print("첫번째 대답:", res1['output'])

# 두 번째 대화 (첫 번째 대화를 넣지 않음)
res2 = agent_executor.invoke({
    "input": "내 이름이 뭐였지?",
    "chat_history": []
})
print("두번째 대답:", res2['output'])
print("-> 결과: 매 턴마다 chat_history를 챙겨주지 않으면, 에이전트는 직전에 무슨 일이 있었는지, 자기가 무슨 툴을 썼었는지 싹 까먹습니다.")

=== [1. 일반 Agent 테스트] ===
첫번째 대답: 당신의 이름 '슈퍼맨'은 3글자입니다.
두번째 대답: 죄송하지만, 당신의 이름을 알 수 없습니다. 알려주시면 도와드릴 수 있습니다!
-> 결과: 매 턴마다 chat_history를 챙겨주지 않으면, 에이전트는 직전에 무슨 일이 있었는지, 자기가 무슨 툴을 썼었는지 싹 까먹습니다.


## 6. 실험 2: RunnableWithMessageHistory로 기억 장치 부착
대화 기록을 저장할 메모리 저장소를 만들고, 에이전트에 씌워줍니다.

In [4]:
print("\n=== [2. 기억하는 Agent 테스트] ===")

# 세션 ID(대화방 번호)별로 대화 기록을 저장할 딕셔너리
store = {}

# 특정 세션 ID의 메모리 객체를 가져오는 함수
def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

# 기존 AgentExecutor 겉에 '자동 메모리 관리자'를 씌웁니다.
agent_with_memory = RunnableWithMessageHistory(
    agent_executor,
    get_session_history,
    input_messages_key="input", # 사용자 질문이 들어가는 변수명
    history_messages_key="chat_history", # 대화 기록이 들어가는 변수명 (프롬프트의 Placeholder와 일치해야 함)
)

# config를 통해 '어느 대화방'인지 명시합니다.
config = {"configurable": {"session_id": "room-1"}}

print("[Turn 1]")
res3 = agent_with_memory.invoke(
    {"input": "안녕? 내 이름은 '배트맨'이야. 내 이름 글자 수가 몇 개지?"},
    config=config
)
print("대답:", res3['output'])

print("\n[Turn 2]")
res4 = agent_with_memory.invoke(
    {"input": "방금 그 숫자에 10을 더하면 얼마야? 그리고 내 이름은 뭐였지?"},
    config=config
)
print("대답:", res4['output'])
print("-> 결과: 완벽합니다! 과거의 맥락(배트맨)과, 직전에 툴을 사용해 얻어낸 결과(글자 수 3)를 모두 기억하고 다음 행동에 반영합니다.")


=== [2. 기억하는 Agent 테스트] ===
[Turn 1]


c:\Users\USER\Desktop\llm-study-lab\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3748: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


대답: 안녕, 배트맨! 너의 이름은 3글자야.

[Turn 2]
대답: 너의 이름은 '배트맨'이고, 글자 수는 3개야. 3에 10을 더하면 13이야.
-> 결과: 완벽합니다! 과거의 맥락(배트맨)과, 직전에 툴을 사용해 얻어낸 결과(글자 수 3)를 모두 기억하고 다음 행동에 반영합니다.


## 7. 결과 해석

1. **`MessagesPlaceholder`**: 프롬프트 안에 대화 기록이 쏙 들어갈 수 있도록 만들어주는 '빈 바구니'입니다.
2. **`RunnableWithMessageHistory`**: 우리가 일일이 사용자 질문, 툴 결과, LLM 응답을 리스트에 `.append()` 하던 귀찮은 작업을 완전히 자동화해주는 래퍼(Wrapper) 클래스입니다.
3. **세션 분리**: `session_id`를 다르게 주면(`room-2`), A 사용자와 B 사용자의 대화가 섞이지 않는 멀티 유저 챗봇을 쉽게 구현할 수 있습니다.

## 8. Notion 실험 로그

### 발견한 문제점 (또는 차이점)
* `AgentExecutor` 자체는 실행 루프만 담당하지 메모리 기능이 없었으나, 래퍼 클래스를 씌움으로써 교안에 나온 **Agent + Memory** 아키텍처가 완성됨.
* 하지만 이 구조에서는 Agent가 *언제, 어떻게* 행동할지 세밀하게 통제하기 어려움. (가령, "특정 툴은 꼭 관리자의 승인을 받고 실행해라" 같은 로직을 짤 수 없음)

### 다음 개선 방향
* 다음 노트북에서는 블랙박스 같은 `AgentExecutor`를 벗어던지고, 최신 실무 기술인 **LangGraph**를 도입.
* 노드와 엣지로 Agent의 흐름을 직접 설계하고, 교안의 핵심 요소 중 하나인 **Guardrails(안전 장치, Human-in-the-loop)**를 구현해 볼 차례임.